In [2]:
import os
os.environ["PYTHONHASHSEED"] = "0"
os.environ["OMP_NUM_THREADS"] = "1"
import tqdm
import json
import joblib
import datetime
from abc import ABC, abstractmethod

import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
# from sklearn.base import BaseEstimator, clone
from sklearn.compose import ColumnTransformer

from sklearn.metrics import mean_squared_error, mean_absolute_error, pairwise_distances
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans

from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from scipy.stats import norm

import catboost as cb

from utils import load_config
from src.helpers import (
    ContiguousGroupKFold, 
    ContiguousTimeSeriesSplit,
    custom_log_likelihood
)

from src.model import MultiOutputRegressor, create_gbt_model, GBTEnsembleRegressor

In [ ]:
save_path = 'path/to/saved/model'
SPATIO_TEMPORAL = False

In [ ]:
model = GBTEnsembleRegressor(use_priors = True, enforce_location = False, spatio_temporal_split = SPATIO_TEMPORAL)
model.load_model(save_path, create_gbt_model, GBT_MODEL_PARAMS)

### Feature Importances

In [ ]:
feat_importances = model.get_feature_importances_()

In [ ]:
plt.style.use('ggplot')

# Customize grid lines: white dashed lines
plt.rcParams['grid.color'] = 'white'
plt.rcParams['grid.linestyle'] = '--'   # dashed lines
plt.rcParams['grid.linewidth'] = 0.8   # adjust thickness if you want
plt.rcParams['axes.grid'] = True

In [ ]:
plt.figure(figsize = (15, 6))
sns.barplot(feat_importances.sort_values('importances', ascending = False).iloc[:50, :], x = 'features', y = 'importances')
plt.tick_params(rotation = 90)
plt.yscale('log')
plt.ylabel('Log importances')
plt.title('Global Feature Importance')
plt.savefig('./assets/images/global_feature_importance.png', bbox_inches = 'tight')

#### Get Per Fold Feature Importances

In [ ]:
raw_importances = model.get_raw_feature_importances_()
feature_names = gbt_model.get_feature_names_()

In [ ]:
# Dictionary to hold feature data
fold_tables = {}
feature_ranks = {}

# Build fold-wise top 10 importance with rank
for i in range(5):
    fold_id = i
    start = 60 * fold_id
    end = start + 60
    fold_importances = np.stack(raw_importances[start:end]).reshape([4, 15, -1])
    
    avg_importances = fold_importances.mean(axis=(0, 1))
    imp_df = pd.DataFrame({
        'features': feature_names,
        'importances': avg_importances
    }).sort_values('importances', ascending=False).reset_index(drop=True)

    top_features = imp_df.iloc[:10]
    fold_col = f'Fold {i+1}'
    fold_tables[fold_col] = {}
    
    for rank, row in top_features.iterrows():
        feature = row['features']
        val = f"{round(row['importances'], 2)} ({rank + 1})"
        fold_tables[fold_col][feature] = val
        
        # Track rank for sorting
        if feature not in feature_ranks:
            feature_ranks[feature] = []
        feature_ranks[feature].append(rank + 1)

# Sort features by mean rank (lower = more important)
sorted_features = sorted(
    feature_ranks.items(),
    key=lambda x: np.mean(x[1])
)
sorted_feature_names = [f[0] for f in sorted_features]

# Generate LaTeX
print("\\begin{tabular}{lccccc}")
print("Feature & Fold 1 & Fold 2 & Fold 3 & Fold 4 & Fold 5 \\\\ \\hline")

for feature in sorted_feature_names:
    latex_feature = feature.replace("_", "\\_")
    row = [latex_feature]
    for fold in ['Fold 1', 'Fold 2', 'Fold 3', 'Fold 4', 'Fold 5']:
        row.append(fold_tables.get(fold, {}).get(feature, ""))
    print(" & ".join(row) + " \\\\")

print("\\end{tabular}")


#### Get per week feature importances

In [ ]:
# Reshape into (4 weeks, 75 models, num_features)
week_importances = np.stack(raw_importances).reshape([4, 75, -1])

# Store results
week_tables = {}
feature_ranks = {}

# Process each week
for i in range(4):
    week_name = f"Week {i+1}"
    avg_importances = week_importances[i].mean(axis=0)

    imp_df = pd.DataFrame({
        'features': feature_names,
        'importances': avg_importances
    }).sort_values('importances', ascending=False).reset_index(drop=True)

    top_features = imp_df.iloc[:10]
    week_tables[week_name] = {}

    for rank, row in top_features.iterrows():
        feature = row['features']
        val = f"{round(row['importances'], 2)} ({rank + 1})"
        week_tables[week_name][feature] = val

        if feature not in feature_ranks:
            feature_ranks[feature] = []
        feature_ranks[feature].append(rank + 1)

# Sort features by average rank
sorted_features = sorted(feature_ranks.items(), key=lambda x: np.mean(x[1]))
sorted_feature_names = [f[0] for f in sorted_features]

# Print LaTeX table
print("\\begin{tabular}{lcccc}")
print("Feature & Week 1 & Week 2 & Week 3 & Week 4 \\\\ \\hline")

for feature in sorted_feature_names:
    latex_feature = feature.replace("_", "\\_")
    row = [latex_feature]
    for week in ['Week 1', 'Week 2', 'Week 3', 'Week 4']:
        row.append(week_tables.get(week, {}).get(feature, ""))
    print(" & ".join(row) + " \\\\")

print("\\end{tabular}")
`

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(12, 15))
ax = ax.flatten()

for i in range(5):
    fold_id = i
    start = 60 * fold_id
    end = start + 60
    
    # Assuming raw_importances[start:end] is a list/array of arrays, stack and reshape
    fold_importances = np.stack(raw_importances[start:end])
    
    # Confirm shape and average over appropriate axes
    # Adjust axes as needed (depends on shape of fold_importances)
    # Here assuming fold_importances shape is (60, some_dim)
    # For demonstration, just mean over axis 0
    mean_importance = fold_importances.mean(axis=0)
    
    imp_df = pd.DataFrame({'features': feature_names, 'importances': mean_importance})
    imp_df_sorted = imp_df.sort_values('importances', ascending=False).head(10)
    
    sns.barplot(data=imp_df_sorted, x='importances', y='features', ax=ax[i], palette='viridis')
    ax[i].set_title(f'Fold {i+1}')
    ax[i].set_xlabel('Importance')
    ax[i].set_ylabel('Feature')
    
plt.tight_layout()
plt.show()


### Performance

In [ ]:
print('mean_nll:', np.mean(list(model.results.values())))
print('mean_empirical_coverage:', np.mean(model.coverage))

In [ ]:
means = []
print("\\begin{tabular}{lcccc}")
print("\\hline")
print("Fold & Week 1 & Week 2 & Week 3 & Week 4 \\\\")
print("\\hline")
for i in range(5):
    f_preds = model.oof_preds[i]['y_pred']
    f_true = model.oof_preds[i]['y_true']
    mae = (f_preds - f_true).abs().mean()
    print(f"fold\\_{i+1} & {mae.iloc[0]:.3f} & {mae.iloc[1]:.3f} & {mae.iloc[2]:.3f} & {mae.iloc[3]:.3f} \\\\")
    means.append(mae)
print("\\hline")
print("\\end{tabular}")
print("\\caption{}")
print("\\label{}")
means = pd.DataFrame(means)


#### Run if SpatioTemporal Split

In [ ]:
if SPATIO_TEMPORAL:
    spatio_means = []
    temporal_means = []
    
    print("\\begin{tabular}{lcccc}")
    print("\\hline")
    print("Split & Week 1 & Week 2 & Week 3 & Week 4 \\\\")
    print("\\hline")
    
    held_out_stations = ['6119040', '6939050', '6139921', '6139790', '5661000', '5685000']
    
    for i in range(5):
        X = model.oof_preds[i]['X'].reset_index(drop=True)
        y_true = model.oof_preds[i]['y_true'].reset_index(drop=True)
        y_pred = model.oof_preds[i]['y_pred']#.reset_index(drop=True)
    
        # Absolute Error
        abs_error = (y_pred - y_true).abs()
        abs_error['station_code'] = X['station_code']
        y_true['station_code'] = X['station_code']
    
        # Group true values and errors by station
        grouped_mae = abs_error.groupby('station_code').mean()
        grouped_true = y_true.groupby('station_code').mean()
    
        # Normalize MAE by mean true values (station-level)
        normalized_mae = grouped_mae / grouped_true.replace(0, np.nan)
    
        # Separate held-out and in-fold stations
        spatio = normalized_mae.loc[normalized_mae.index.isin(held_out_stations)].mean()
        temporal = normalized_mae.loc[~normalized_mae.index.isin(held_out_stations)].mean()
    
        spatio_means.append(spatio)
        temporal_means.append(temporal)
    
    # Aggregate across folds
    spatio_means = pd.DataFrame(spatio_means).mean()
    temporal_means = pd.DataFrame(temporal_means).mean()
    
    # Print LaTeX row
    print(f"Temporal\\_split & {temporal_means[0]:.3f} & {temporal_means[1]:.3f} & {temporal_means[2]:.3f} & {temporal_means[3]:.3f} \\\\")
    print(f"Spatio\\_temporal\\_split & {spatio_means[0]:.3f} & {spatio_means[1]:.3f} & {spatio_means[2]:.3f} & {spatio_means[3]:.3f} \\\\")
    print("\\hline")
    print("\\end{tabular}")
    print("\\caption{Station-normalized Mean Absolute Error across Weeks for Temporal and Spatiotemporal Splits}")
    print("\\label{tab:station_normalized_mae}")


In [ ]:
if SPATIO_TEMPORAL:
    spatio_means = []
    temporal_means = []
    
    print("\\begin{tabular}{lcccc}")
    print("\\hline")
    print("Split & Week 1 & Week 2 & Week 3 & Week 4 \\\\")
    print("\\hline")
    
    held_out_stations = ['6119040', '6939050', '6139921', '6139790', '5661000', '5685000']
    
    for i in range(5):
        X = model.oof_preds[i]['X'].reset_index(drop=True)
        y_true = model.oof_preds[i]['y_true'].reset_index(drop=True)
        y_lower = model.oof_preds[i]['y_lower'].reset_index(drop=True)
        y_upper = model.oof_preds[i]['y_upper'].reset_index(drop=True)
    
        # Compute interval width
        interval_width = (y_upper - y_lower).abs()
        interval_width['station_code'] = X['station_code']
        y_true['station_code'] = X['station_code']
        
        # Group by station and compute mean interval width and mean true value
        grouped_width = interval_width.groupby('station_code').mean()
        grouped_true = y_true.groupby('station_code').mean()
    
        # Normalize interval width by true value (station-level)
        normalized_interval = pd.DataFrame(grouped_width.values / grouped_true.values, index = grouped_width.index) #.replace(0, np.nan)
    
        # Split into spatiotemporal and temporal subsets
        spatio = normalized_interval.loc[normalized_interval.index.isin(held_out_stations)].mean()
        temporal = normalized_interval.loc[~normalized_interval.index.isin(held_out_stations)].mean()
    
        spatio_means.append(spatio)
        temporal_means.append(temporal)
    
    # Aggregate across folds
    spatio_means = pd.DataFrame(spatio_means).mean()
    temporal_means = pd.DataFrame(temporal_means).mean()
    
    # Output LaTeX-formatted table rows
    print(f"Temporal\\_split & {temporal_means[0]:.3f} & {temporal_means[1]:.3f} & {temporal_means[2]:.3f} & {temporal_means[3]:.3f} \\\\")
    print(f"Spatio\\_temporal\\_split & {spatio_means[0]:.3f} & {spatio_means[1]:.3f} & {spatio_means[2]:.3f} & {spatio_means[3]:.3f} \\\\")
    print("\\hline")
    print("\\end{tabular}")
    print("\\caption{Station-normalized Prediction Interval Width across Weeks for Temporal and Spatiotemporal Splits}")
    print("\\label{tab:normalized_interval_width}")


### Frugality

In [ ]:
def get_size(start_path = '.'):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(start_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            # skip if it is symbolic link
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)

    return total_size


In [ ]:
size_bytes = get_size(save_path)
print(size_bytes)
print(save_path)
size_megabytes = size_bytes / (1024 ** 2)

print(f"Model size: {size_megabytes:.2f} MB")
